# Notebook 01: Orquestación de Ingestión Multifuente y Landing Inmutable
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Bronze Landing  
**Estándar**: SWEBOK v4 / DAMA-DMBOK 2  

---
### Objetivo Operativo
Consumir de forma resiliente y paginada las **12 fuentes gubernamentales oficiales** (1 ICA Azure PowerBI + 7 IDEAM SODA + 2 DANE/SIPSA SODA + 2 UPRA SODA), persistiendo los datos crudos en formato Parquet comprimido con Snappy bajo verificación de hash SHA-256.

In [1]:
# Celda 1: Configuración de Entorno e Importaciones Desacopladas de src/
import sys
from pathlib import Path

# Asegurar resolución del paquete raíz src
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.ingestion.socrata_client import SocrataClient
from src.ingestion.ica_client import ICAClient
from src.ingestion.landing_manager import LandingManager

logger = get_logger("notebooks.01_ingestion")
logger.info("Entorno inicializado. Modo: %s | Base Dir: %s", settings.ENV, settings.BASE_DIR)

# Instanciar clientes y gestor de almacenamiento Bronze
socrata = SocrataClient()
ica = ICAClient()
landing = LandingManager()

2026-10-07T19:28:18-0500 | INFO     | agrostats.notebooks.01_ingestion:<module>:17 | Entorno inicializado. Modo: development | Base Dir: C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp
2026-10-07T19:28:18-0500 | WARNING  | agrostats.ingestion.socrata_client:__init__:34 | SocrataClient inicializado sin App-Token (cuota pública limitada)
2026-10-07T19:28:18-0500 | INFO     | agrostats.ingestion.ica_client:__init__:37 | ICAClient configurado con ResourceKey: a7581d79...


In [2]:
# Celda 2: Verificación de Salud (Healthcheck) de los Endpoints SODA 2.0
print("=== REVISIÓN DE CONECTIVIDAD DE LOS 11 RECURSOS SOCRATA ===")
health_report = {}
for key, dataset in settings.SOCRATA_DATASETS.items():
    is_alive = socrata.check_health(dataset.resource_id)
    health_report[dataset.resource_id] = is_alive
    status_str = "ONLINE [200 OK]" if is_alive else "OFFLINE / RATE LIMIT"
    print(f"[{dataset.entidad}] {dataset.nombre} ({dataset.resource_id}): {status_str}")

total_online = sum(1 for v in health_report.values() if v)
print(f"\nResumen: {total_online}/{len(health_report)} endpoints activos.")

=== REVISIÓN DE CONECTIVIDAD DE LOS 11 RECURSOS SOCRATA ===
[IDEAM] Telemetría Sensores Realtime (57sv-p2fu): ONLINE [200 OK]
[IDEAM] Pluviometría y Precipitación (s54a-sgyg): ONLINE [200 OK]
[IDEAM] Temperatura Ambiente 2m (sbwg-7ju4): ONLINE [200 OK]
[IDEAM] Temperatura Máxima (ccvq-rp9s): ONLINE [200 OK]
[IDEAM] Temperatura Mínima (afdg-3zpb): ONLINE [200 OK]
[IDEAM] Radiación Solar Global (rv9s-8nv6): ONLINE [200 OK]
[IDEAM] Normales Climatológicas & SPI (nsz2-kzcq): ONLINE [200 OK]
[DANE] SIPSA Insumos Agrícolas (gwbi-fnzs): ONLINE [200 OK]
[DANE] SIPSA Alimentos Balanceados (ABA) (wgj6-cvyj): ONLINE [200 OK]
[UPRA] Exportaciones & Bioinsumos (gaic-b8aw): ONLINE [200 OK]
[UPRA] Evaluaciones Agropecuarias (EVA) (2pnw-mmge): ONLINE [200 OK]

Resumen: 11/11 endpoints activos.


In [3]:
# Celda 3: Extracción de Datasets Climáticos IDEAM (7 Endpoints)
# Para ejecución de desarrollo, limitamos a batch de prueba (ej. 5,000 filas)
ideam_datasets = [
    ("ideam_telemetria", "telemetria_sensores"),
    ("ideam_pluviometria", "pluviometria_estaciones"),
    ("ideam_temp_media", "temperatura_media"),
    ("ideam_temp_maxima", "temperatura_maxima"),
    ("ideam_temp_minima", "temperatura_minima"),
    ("ideam_radiacion", "radiacion_solar"),
    ("ideam_normales_spi", "normales_spi"),
]

for cfg_key, target_name in ideam_datasets:
    cfg = settings.SOCRATA_DATASETS[cfg_key]
    print(f"Descargando {cfg.nombre} ({cfg.resource_id})...")
    df_ideam = socrata.fetch_dataframe(resource_id=cfg.resource_id, max_records=5000)
    if not df_ideam.empty:
        meta = landing.save_raw_dataset(df_ideam, source_category="ideam", filename=f"{target_name}.parquet")
        print(f" -> Guardado: {meta['filename']} | Filas: {meta['rows']} | SHA-256: {meta['sha256'][:12]}...")

Descargando Telemetría Sensores Realtime (57sv-p2fu)...


2026-10-07T19:28:22-0500 | INFO     | agrostats.ingestion.socrata_client:fetch_dataframe:115 | Dataset 57sv-p2fu consolidado en DataFrame con 5000 filas
2026-10-07T19:28:22-0500 | INFO     | agrostats.ingestion.landing_manager:save_raw_dataset:81 | Archivo inmutable existente en C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\RAW\ideam\telemetria_sensores.parquet (SHA-256: 8abdda45af...). Omitiendo sobrescritura.


 -> Guardado: telemetria_sensores.parquet | Filas: 5000 | SHA-256: 8abdda45afa5...
Descargando Pluviometría y Precipitación (s54a-sgyg)...


2026-10-07T19:28:23-0500 | INFO     | agrostats.ingestion.socrata_client:fetch_dataframe:115 | Dataset s54a-sgyg consolidado en DataFrame con 5000 filas
2026-10-07T19:28:23-0500 | INFO     | agrostats.ingestion.landing_manager:save_raw_dataset:81 | Archivo inmutable existente en C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\RAW\ideam\pluviometria_estaciones.parquet (SHA-256: 0251f61272...). Omitiendo sobrescritura.


 -> Guardado: pluviometria_estaciones.parquet | Filas: 5000 | SHA-256: 0251f612727b...
Descargando Temperatura Ambiente 2m (sbwg-7ju4)...


...

 -> Guardado: temperatura_minima.parquet | Filas: 5000 | SHA-256: f05a6f9d8739...
Descargando Radiación Solar Global (rv9s-8nv6)...


2026-10-07T19:28:25-0500 | INFO     | agrostats.ingestion.socrata_client:fetch_dataframe:115 | Dataset rv9s-8nv6 consolidado en DataFrame con 169 filas
2026-10-07T19:28:25-0500 | INFO     | agrostats.ingestion.landing_manager:save_raw_dataset:81 | Archivo inmutable existente en C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\RAW\ideam\radiacion_solar.parquet (SHA-256: 0b1e92b50e...). Omitiendo sobrescritura.


 -> Guardado: radiacion_solar.parquet | Filas: 169 | SHA-256: 0b1e92b50ec3...
Descargando Normales Climatológicas & SPI (nsz2-kzcq)...


2026-10-07T19:28:26-0500 | INFO     | agrostats.ingestion.socrata_client:fetch_dataframe:115 | Dataset nsz2-kzcq consolidado en DataFrame con 5000 filas
2026-10-07T19:28:26-0500 | INFO     | agrostats.ingestion.landing_manager:save_raw_dataset:81 | Archivo inmutable existente en C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\RAW\ideam\normales_spi.parquet (SHA-256: 648c291565...). Omitiendo sobrescritura.


 -> Guardado: normales_spi.parquet | Filas: 5000 | SHA-256: 648c2915658b...


In [4]:
# Celda 4: Extracción de Insumos y Alimentos SIPSA / DANE
sipsa_datasets = [
    ("sipsa_insumos", "sipsa_insumos_precios"),
    ("sipsa_aba", "sipsa_alimentos_balanceados"),
]

for cfg_key, target_name in sipsa_datasets:
    cfg = settings.SOCRATA_DATASETS[cfg_key]
    print(f"Descargando {cfg.nombre} ({cfg.resource_id})...")
    df_sipsa = socrata.fetch_dataframe(resource_id=cfg.resource_id, max_records=5000)
    if not df_sipsa.empty:
        meta = landing.save_raw_dataset(df_sipsa, source_category="dane", filename=f"{target_name}.parquet")
        print(f" -> Guardado: {meta['filename']} | Filas: {meta['rows']} | SHA-256: {meta['sha256'][:12]}...")

Descargando SIPSA Insumos Agrícolas (gwbi-fnzs)...


2026-10-07T19:28:26-0500 | INFO     | agrostats.ingestion.socrata_client:fetch_dataframe:115 | Dataset gwbi-fnzs consolidado en DataFrame con 93 filas
2026-10-07T19:28:26-0500 | INFO     | agrostats.ingestion.landing_manager:save_raw_dataset:81 | Archivo inmutable existente en C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\RAW\dane\sipsa_insumos_precios.parquet (SHA-256: e66e97a050...). Omitiendo sobrescritura.


 -> Guardado: sipsa_insumos_precios.parquet | Filas: 93 | SHA-256: e66e97a050fa...
Descargando SIPSA Alimentos Balanceados (ABA) (wgj6-cvyj)...


2026-10-07T19:28:27-0500 | INFO     | agrostats.ingestion.socrata_client:fetch_dataframe:115 | Dataset wgj6-cvyj consolidado en DataFrame con 68 filas
2026-10-07T19:28:27-0500 | INFO     | agrostats.ingestion.landing_manager:save_raw_dataset:81 | Archivo inmutable existente en C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\RAW\dane\sipsa_alimentos_balanceados.parquet (SHA-256: d2a6efbacf...). Omitiendo sobrescritura.


 -> Guardado: sipsa_alimentos_balanceados.parquet | Filas: 68 | SHA-256: d2a6efbacf5a...


In [5]:
# Celda 5: Extracción de UPRA (EVA y Exportaciones Agropecuarias)
upra_datasets = [
    ("upra_exportaciones", "upra_exportaciones_bioinsumos"),
    ("upra_eva", "upra_eva_historico"),
]

for cfg_key, target_name in upra_datasets:
    cfg = settings.SOCRATA_DATASETS[cfg_key]
    print(f"Descargando {cfg.nombre} ({cfg.resource_id})...")
    df_upra = socrata.fetch_dataframe(resource_id=cfg.resource_id, max_records=5000)
    if not df_upra.empty:
        meta = landing.save_raw_dataset(df_upra, source_category="upra", filename=f"{target_name}.parquet")
        print(f" -> Guardado: {meta['filename']} | Filas: {meta['rows']} | SHA-256: {meta['sha256'][:12]}...")

Descargando Exportaciones & Bioinsumos (gaic-b8aw)...


2026-10-07T19:28:27-0500 | INFO     | agrostats.ingestion.socrata_client:fetch_dataframe:115 | Dataset gaic-b8aw consolidado en DataFrame con 5000 filas
2026-10-07T19:28:27-0500 | INFO     | agrostats.ingestion.landing_manager:save_raw_dataset:81 | Archivo inmutable existente en C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\RAW\upra\upra_exportaciones_bioinsumos.parquet (SHA-256: 0f8c5f9596...). Omitiendo sobrescritura.


 -> Guardado: upra_exportaciones_bioinsumos.parquet | Filas: 5000 | SHA-256: 0f8c5f9596fe...
Descargando Evaluaciones Agropecuarias (EVA) (2pnw-mmge)...


2026-10-07T19:28:28-0500 | INFO     | agrostats.ingestion.socrata_client:fetch_dataframe:115 | Dataset 2pnw-mmge consolidado en DataFrame con 5000 filas
2026-10-07T19:28:28-0500 | INFO     | agrostats.ingestion.landing_manager:save_raw_dataset:114 | Dataset guardado en Bronze: upra_eva_historico.parquet (5000 filas, 69533 bytes, SHA256: 4ca59e4fc8...)


 -> Guardado: upra_eva_historico.parquet | Filas: 5000 | SHA-256: 4ca59e4fc836...


In [6]:
# Celda 6: Extracción Pecuaria ICA (Azure PowerBI QueryData API)
print("Consultando Backend PowerBI QueryData del ICA...")
try:
    df_ica = ica.fetch_censo_dataframe()
    if not df_ica.empty:
        meta_ica = landing.save_raw_dataset(df_ica, source_category="ica", filename="censo_pecuario.parquet")
        print(f" -> Guardado ICA: {meta_ica['filename']} | Filas: {meta_ica['rows']} | SHA-256: {meta_ica['sha256'][:12]}...")
    else:
        print(" -> Respuesta recibida de ICA pero sin registros tabulares parseables en el payload actual.")
except Exception as exc:
    print(f"Aviso: Endpoint de ICA no accesible en este instante o requiere red institucional: {exc}")

2026-10-07T19:28:28-0500 | WARNING  | agrostats.ingestion.base_client:log_it:64 | Retrying src.ingestion.base_client.BaseAPIClient.request_with_retry.<locals>._execute in 1 seconds as it raised ConnectionError: HTTPSConnectionPool(host='wabi-us-north-central-a-primary-redirect.analysis.windows.net', port=443): Max retries exceeded with url: /powerbi/api/v1.0/public/reports/querydata?synchronous=true (Caused by NameResolutionError("HTTPSConnection(host='wabi-us-north-central-a-primary-redirect.analysis.windows.net', port=443): Failed to resolve 'wabi-us-north-central-a-primary-redirect.analysis.windows.net' ([Errno 11001] getaddrinfo failed)")).


Consultando Backend PowerBI QueryData del ICA...


2026-10-07T19:28:29-0500 | WARNING  | agrostats.ingestion.base_client:log_it:64 | Retrying src.ingestion.base_client.BaseAPIClient.request_with_retry.<locals>._execute in 2 seconds as it raised ConnectionError: HTTPSConnectionPool(host='wabi-us-north-central-a-primary-redirect.analysis.windows.net', port=443): Max retries exceeded with url: /powerbi/api/v1.0/public/reports/querydata?synchronous=true (Caused by NameResolutionError("HTTPSConnection(host='wabi-us-north-central-a-primary-redirect.analysis.windows.net', port=443): Failed to resolve 'wabi-us-north-central-a-primary-redirect.analysis.windows.net' ([Errno 11001] getaddrinfo failed)")).
2026-10-07T19:28:31-0500 | WARNING  | agrostats.ingestion.base_client:log_it:64 | Retrying src.ingestion.base_client.BaseAPIClient.request_with_retry.<locals>._execute in 4 seconds as it raised ConnectionError: HTTPSConnectionPool(host='wabi-us-north-central-a-primary-redirect.analysis.windows.net', port=443): Max retries exceeded with url: /pow

Aviso: Endpoint de ICA no accesible en este instante o requiere red institucional: HTTPSConnectionPool(host='wabi-us-north-central-a-primary-redirect.analysis.windows.net', port=443): Max retries exceeded with url: /powerbi/api/v1.0/public/reports/querydata?synchronous=true (Caused by NameResolutionError("HTTPSConnection(host='wabi-us-north-central-a-primary-redirect.analysis.windows.net', port=443): Failed to resolve 'wabi-us-north-central-a-primary-redirect.analysis.windows.net' ([Errno 11001] getaddrinfo failed)"))


In [7]:
# Celda 7: Auditoría Criptográfica de la Capa Bronze (Verificación SHA-256)
print("=== REPORTE DE AUDITORÍA E INTEGRIDAD BRONZE ===")
manifest = landing.load_manifest()
integrity_check = landing.verify_integrity()

for dataset_key, is_intact in integrity_check.items():
    meta = manifest["datasets"].get(dataset_key, {})
    status = "INTEGRO (OK)" if is_intact else "CORRUPTO / MODIFICADO (ERROR)"
    print(f"[{status}] {dataset_key} | Filas: {meta.get('rows', 0)} | SHA-256: {meta.get('sha256', '')[:16]}...")

print(f"\nÚltima actualización del manifiesto: {manifest.get('updated_at')}")

=== REPORTE DE AUDITORÍA E INTEGRIDAD BRONZE ===
[INTEGRO (OK)] ideam/telemetria_sensores.parquet | Filas: 5000 | SHA-256: 8abdda45afa599b2...
[INTEGRO (OK)] ideam/pluviometria_estaciones.parquet | Filas: 5000 | SHA-256: 0251f612727baeb6...
[INTEGRO (OK)] ideam/temperatura_media.parquet | Filas: 5000 | SHA-256: 6cbc73167b563509...
[INTEGRO (OK)] ideam/temperatura_maxima.parquet | Filas: 5000 | SHA-256: ae8d28b211492573...
[INTEGRO (OK)] ideam/temperatura_minima.parquet | Filas: 5000 | SHA-256: f05a6f9d87399cc0...
[INTEGRO (OK)] ideam/radiacion_solar.parquet | Filas: 169 | SHA-256: 0b1e92b50ec31506...
[INTEGRO (OK)] ideam/normales_spi.parquet | Filas: 5000 | SHA-256: 648c2915658b9a79...
[INTEGRO (OK)] dane/sipsa_insumos_precios.parquet | Filas: 93 | SHA-256: e66e97a050faf67b...
[INTEGRO (OK)] dane/sipsa_alimentos_balanceados.parquet | Filas: 68 | SHA-256: d2a6efbacf5a9c5f...
[INTEGRO (OK)] upra/upra_exportaciones_bioinsumos.parquet | Filas: 5000 | SHA-256: 0f8c5f9596fe6ce8...
[INTEGRO (